# 3. Reinforcement Learning con Verifiable Rewards (GRPO)
**HLT 2026 - Reasoning with Verifiable Rewards**

In questo notebook implementiamo l'addestramento con **Group Relative Policy Optimization (GRPO)** ispirato a **DeepSeekMath** e **DeepSeek-R1-Zero** usando Hugging Face TRL:
1. **Zero Critic**: Elimina il Critic di PPO, campionando $G$ generazioni per prompt e calcolando i vantaggi normalizzati all'interno del gruppo.
2. **Verifiable Rewards**: Tre funzioni di ricompensa basate su regole (Accuracy, Format, Syntax).
3. **PEFT (LoRA)**: Addestramento leggero che si adatta comodamente alla VRAM di una singola GPU gratuita (Kaggle T4 o Colab T4).

> ⚠️ **Requisito hardware**: **Richiede GPU** (Colab T4 o Kaggle 2xT4). L'RL su CPU non e praticabile per via dei numerosi rollout paralleli.

## 1. Installazione librerie (su Colab o Kaggle)

In [ ]:
!pip install -q "trl>=0.14.0" "transformers>=4.48.0" peft accelerate datasets bitsandbytes

## 2. Inizializzazione Hardware & Configurazione Modello

In [ ]:
import os
import ast
import re
import json
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig

assert torch.cuda.is_available(), "GPU non rilevata! Attiva la GPU su Runtime -> Change runtime type -> T4 GPU"
print(f"GPU attiva: {torch.cuda.get_device_name(0)}")

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

## 3. Preparazione Dataset per GRPOTrainer

In [ ]:
train_file = "../data/train_countdown.jsonl"
if os.path.exists(train_file):
    with open(train_file, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f]
else:
    # Dataset di fallback
    data = [
        {"numbers": [84, 83, 34, 72], "target": 39},
        {"numbers": [25, 5, 2, 8], "target": 60},
        {"numbers": [12, 3, 4, 7], "target": 35},
        {"numbers": [50, 2, 3, 5], "target": 105}
    ] * 250

train_prompts = []
for d in data:
    train_prompts.append({
        "prompt": [
            {"role": "system", "content": "You are a helpful mathematical assistant."},
            {"role": "user", "content": f"Using the numbers {d['numbers']}, create an equation that equals {d['target']}. You can use basic arithmetic operations (+, -, *, /) and each number can be used at most once. Show your reasoning in <think>...</think> and place the final equation in <answer>...</answer>."}
        ],
        "numbers": d["numbers"],
        "target": d["target"]
    })

train_dataset = Dataset.from_list(train_prompts)
print(f"Dataset pronto: {len(train_dataset)} esempi per il training.")

## 4. Definizione Funzioni di Reward Verificabili (R1-Zero Style)

Definiamo 3 reward functions secondo le specifiche di DeepSeek-R1-Zero:
1. **`format_reward_func`** (+0.2): rispetto della struttura `<think>...</think><answer>...</answer>`.
2. **`equation_validity_reward_func`** (+0.1): espressione sintatticamente valida con soli numeri consentiti.
3. **`accuracy_reward_func`** (+1.0): correttezza matematica del target.

In [ ]:
def format_reward_func(completions, **kwargs):
    rewards = []
    pattern = r"^<think>.*?</think>s*<answer>.*?</answer>$"
    for comp in completions:
        content = comp[0]["content"] if isinstance(comp, list) else comp
        if re.search(pattern, content.strip(), re.DOTALL | re.IGNORECASE):
            rewards.append(0.2)
        elif "<answer>" in content and "</answer>" in content:
            rewards.append(0.1)
        else:
            rewards.append(0.0)
    return rewards

def equation_validity_reward_func(completions, numbers, **kwargs):
    rewards = []
    for comp, nums in zip(completions, numbers):
        content = comp[0]["content"] if isinstance(comp, list) else comp
        ans_match = re.search(r"<answer>(.*?)</answer>", content, re.DOTALL)
        if not ans_match:
            rewards.append(0.0)
            continue
        expr = ans_match.group(1).split("=")[0].strip()
        try:
            tree = ast.parse(expr, mode='eval')
            used = []
            for n in ast.walk(tree):
                if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)):
                    used.append(int(n.value) if float(n.value).is_integer() else n.value)
                elif hasattr(ast, 'Num') and isinstance(n, ast.Num):
                    used.append(int(n.n) if float(n.n).is_integer() else n.n)
            avail = list(nums)
            valid = True
            for u in used:
                if u in avail: avail.remove(u)
                else: valid = False; break
            rewards.append(0.1 if (valid and len(used) > 0) else 0.0)
        except Exception:
            rewards.append(0.0)
    return rewards

def accuracy_reward_func(completions, numbers, target, **kwargs):
    rewards = []
    for comp, nums, tgt in zip(completions, numbers, target):
        content = comp[0]["content"] if isinstance(comp, list) else comp
        ans_match = re.search(r"<answer>(.*?)</answer>", content, re.DOTALL)
        if not ans_match:
            rewards.append(0.0)
            continue
        expr = ans_match.group(1).split("=")[0].strip()
        try:
            tree = ast.parse(expr, mode='eval')
            used = []
            for n in ast.walk(tree):
                if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)):
                    used.append(int(n.value) if float(n.value).is_integer() else n.value)
                elif hasattr(ast, 'Num') and isinstance(n, ast.Num):
                    used.append(int(n.n) if float(n.n).is_integer() else n.n)
            avail = list(nums)
            valid = True
            for u in used:
                if u in avail: avail.remove(u)
                else: valid = False; break
            if not valid or len(used) == 0:
                rewards.append(0.0)
                continue

            allowed = {
                ast.Add: lambda a, b: a + b,
                ast.Sub: lambda a, b: a - b,
                ast.Mult: lambda a, b: a * b,
                ast.Div: lambda a, b: a / b if b != 0 else None,
                ast.USub: lambda a: -a
            }
            def ev(node):
                if isinstance(node, ast.Expression): return ev(node.body)
                elif isinstance(node, ast.Constant): return float(node.value)
                elif hasattr(ast, 'Num'): return float(node.n)
                elif isinstance(node, ast.UnaryOp):
                    op = allowed.get(type(node.op)); v = ev(node.operand)
                    return op(v) if op and v is not None else None
                elif isinstance(node, ast.BinOp):
                    op = allowed.get(type(node.op)); l = ev(node.left); r = ev(node.right)
                    return op(l, r) if op and l is not None and r is not None else None
                return None

            computed = ev(tree)
            if computed is not None and abs(computed - tgt) < 1e-6:
                rewards.append(1.0)
            else:
                rewards.append(0.0)
        except Exception:
            rewards.append(0.0)
    return rewards

## 5. Configurazione LoRA e GRPOTrainer

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = GRPOConfig(
    output_dir="../models/qwen-countdown-grpo",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4, # Gruppo G=4 per la stima del baseline relativo
    max_completion_length=384,
    max_prompt_length=256,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    bf16=True if torch.cuda.is_bf16_supported() else False,
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none"
)

trainer = GRPOTrainer(
    model=MODEL_ID,
    reward_funcs=[accuracy_reward_func, format_reward_func, equation_validity_reward_func],
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config
)

## 6. Esecuzione Addestramento GRPO

In [ ]:
print("Avvio addestramento GRPO con Verifiable Rewards...")
trainer.train()

trainer.model.save_pretrained("../models/qwen-countdown-grpo-final")
print(" Modello addestrato e pesi LoRA salvati con successo!")

## 7. Confronto Finale delle 3 Metodologie (per la Relazione)

In [ ]:
if os.path.exists("../results/baselines.json"):
    with open("../results/baselines.json") as f:
        base = json.load(f)
    print("=== TABELLA COMPARATIVA PROGETTO ===")
    print(f"1. Zero-Shot Direct:        {base['zero_shot_accuracy']:.2f}% (Avg {base['zero_shot_tokens']:.1f} tokens)")
    print(f"2. Chain-of-Thought (CoT):  {base['cot_accuracy']:.2f}% (Avg {base['cot_tokens']:.1f} tokens)")
    print("3. GRPO (RL Model):          [Calcolato sul modello addestrato]")